<a href="https://colab.research.google.com/github/AyushNegi785/depression-detection-bert/blob/main/bertcustom.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

In [ ]:
# --- Configuration and Hyperparameters ---
MODEL_NAME = 'bert-base-uncased'
INPUT_FILENAME = 'balanced_data.csv'
TEXT_COLUMN = 'text'
LABEL_COLUMN = 'PHQ_Binary' # Using the corrected column name

In [ ]:
# Hyperparameters
MAX_LEN = 128
BATCH_SIZE = 8
EPOCHS = 10 # Train on all data for 10 epochs
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
NUM_WORKERS = 2

In [ ]:
# --- 1. Load Full Dataset ---
print("Loading and preparing data...")
df_full = pd.read_csv(INPUT_FILENAME)
df_full[TEXT_COLUMN] = df_full['question'] + " [SEP] " + df_full['answer']
print(f"Loaded {len(df_full)} samples for final training.")

Loading and preparing data...
Loaded 4266 samples for final training.


In [ ]:
# --- 2. Custom PyTorch Dataset ---
class DepressionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
# --- 3. Dataloader and Model Setup ---
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Create the Dataset
train_dataset = DepressionDataset(
    texts=df_full[TEXT_COLUMN].to_numpy(),
    labels=df_full[LABEL_COLUMN].to_numpy(),
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

# Create the DataLoader
train_data_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    shuffle=True # Shuffle for training
)

# Initialize the Model
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)
model = model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Using device: cuda


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# --- 4. Optimizer and Scheduler ---
optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)
total_steps = len(train_data_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

In [ ]:
# --- 5. Training Function ---
def train_epoch(model, data_loader, optimizer, device, scheduler):
    model = model.train()
    losses = []
    for d in data_loader:
        input_ids = d["input_ids"].to(device)
        attention_mask = d["attention_mask"].to(device)
        labels = d["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        losses.append(loss.item())
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
    return np.mean(losses)

In [ ]:
# --- 6.Evaluation Function ---
def eval_model(model, data_loader, device):
    model = model.eval()
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            labels = d["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            logits = outputs.logits

            preds = torch.argmax(logits, dim=1).cpu().numpy()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds)

    return {
        'labels': all_labels,
        'predictions': all_preds
    }

In [ ]:
# --- 7. Full Training Loop ---
print("--- Starting Final Model Training ---")
for epoch in range(EPOCHS):
    print(f'  Epoch {epoch + 1}/{EPOCHS}')
    avg_train_loss = train_epoch(model, train_data_loader, optimizer, device, scheduler)
    print(f'  Average Training Loss: {avg_train_loss:.4f}')

--- Starting Final Model Training ---
  Epoch 1/10
  Average Training Loss: 0.6837
  Epoch 2/10
  Average Training Loss: 0.5957
  Epoch 3/10
  Average Training Loss: 0.3793
  Epoch 4/10
  Average Training Loss: 0.2152
  Epoch 5/10
  Average Training Loss: 0.1443
  Epoch 6/10
  Average Training Loss: 0.1068
  Epoch 7/10
  Average Training Loss: 0.0892
  Epoch 8/10
  Average Training Loss: 0.0724
  Epoch 9/10
  Average Training Loss: 0.0650
  Epoch 10/10
  Average Training Loss: 0.0630


In [ ]:
# --- 8. Evaluate the Final Model on Training Data ---
print("\n--- Evaluating Final Model on Training Data ---")
eval_data_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    shuffle=False
)

eval_results = eval_model(model, eval_data_loader, device)
final_labels = eval_results['labels']
final_preds = eval_results['predictions']

# Calculate metrics
final_accuracy = accuracy_score(final_labels, final_preds)
final_f1 = f1_score(final_labels, final_preds)
cm = confusion_matrix(final_labels, final_preds)

print(f"Final Training Accuracy: {final_accuracy:.4f}")
print(f"Final Training F1-Score: {final_f1:.4f}")

print('\n--- Final Training Confusion Matrix ---')
print(f"\n          Predicted 0         Predicted 1")
print(f"          (No Depression)   (Depression)")
print(f"Actual 0    {cm[0, 0]:<10}      {cm[0, 1]:<10}")
print(f"(No Depression)")
print(f"Actual 1    {cm[1, 0]:<10}      {cm[1, 1]:<10}")
print(f"(Depression)")


--- Evaluating Final Model on Training Data ---
Final Training Accuracy: 0.9735
Final Training F1-Score: 0.9739

--- Final Training Confusion Matrix ---

          Predicted 0         Predicted 1
          (No Depression)   (Depression)
Actual 0    2043            90        
(No Depression)
Actual 1    23              2110      
(Depression)


In [ ]:
# --- 9. Save the Final Model ---
model_save_path = '/content/drive/MyDrive/splits/final_bert_model.bin'
torch.save(model.state_dict(), model_save_path)
print(f"\nFinal model saved to: {model_save_path}")


Final model saved to: /content/drive/MyDrive/splits/final_bert_model.bin
